In [2]:
import climkern as ck

In [3]:
ctrl, pert = ck.tutorial_data("ctrl"), ck.tutorial_data("pert")

In [4]:
LR, Planck = ck.calc_T_feedbacks(
    ctrl.T, ctrl.TS, ctrl.PS, pert.T, pert.TS, pert.PS, pert.TROP_P, kern="GFDL"
)

In [5]:
# compute global average surface temperature change
dTS_glob_avg = ck.spat_avg(pert.TS - ctrl.TS)

# normalize temperature feedbacks by temperature change and take
# the annual average
print("The global average lapse rate feedback is {val:.2f} W/m^2/K.".format(
    val=(ck.spat_avg(LR)/dTS_glob_avg).mean()))
print("The global average Planck feedback is {val:.2f} W/m^2/K.".format(
    val=(ck.spat_avg(Planck)/dTS_glob_avg).mean()))

The global average lapse rate feedback is -0.41 W/m^2/K.
The global average Planck feedback is -3.12 W/m^2/K.


In [6]:
q_lw,q_sw = ck.calc_q_feedbacks(ctrl.Q,ctrl.T,ctrl.PS,
                                pert.Q,pert.PS,pert.TROP_P,
                                kern="GFDL",method=1)
alb = ck.calc_alb_feedback(ctrl.FSUS,ctrl.FSDS,
                           pert.FSUS,pert.FSDS,
                           kern="GFDL")

print("The global average water vapor feedback is {val:.2f} W/m^2/K.".format(
    val=(ck.spat_avg(q_lw+q_sw)/dTS_glob_avg).mean()))
print("The global average surface albedo feedback is {val:.2f} W/m^2/K."
      .format(
    val=(ck.spat_avg(alb)/dTS_glob_avg).mean()))

The global average water vapor feedback is 1.44 W/m^2/K.
The global average surface albedo feedback is 0.38 W/m^2/K.


In [7]:
dCRE_LW = ck.calc_dCRE_LW(ctrl.FLNT,pert.FLNT,ctrl.FLNTC,pert.FLNTC)
dCRE_SW = ck.calc_dCRE_SW(ctrl.FSNT,pert.FSNT,ctrl.FSNTC,pert.FSNTC)

In [8]:
erf = ck.tutorial_data('ERF')

In [9]:
#_cs means clear-sky
LR_cs,Planck_cs = ck.calc_T_feedbacks(ctrl.T,ctrl.TS,ctrl.PS,
                                pert.T,pert.TS,pert.PS,pert.TROP_P,
                                kern="GFDL",sky="clear-sky")
q_lw_cs,q_sw_cs = ck.calc_q_feedbacks(ctrl.Q,ctrl.T,ctrl.PS,
                                pert.Q,pert.PS,pert.TROP_P,
                                kern="GFDL",method=1,sky="clear-sky")
alb_cs = ck.calc_alb_feedback(ctrl.FSUS,ctrl.FSDS,
                           pert.FSUS,pert.FSDS,
                           kern="GFDL",sky="clear-sky")

In [10]:
cld_lw = ck.calc_cloud_LW(LR + Planck,LR_cs+Planck_cs,q_lw,q_lw_cs,dCRE_LW,
                          erf.erf_lwas,erf.erf_lwcs)
cld_sw = ck.calc_cloud_SW(alb,alb_cs,q_sw,q_sw_cs,dCRE_SW,erf.erf_swas,
                          erf.erf_swcs)

print("The global average SW cloud feedback is {val:.2f} W/m^2/K.".format(
    val=(ck.spat_avg(cld_sw)/dTS_glob_avg).mean()))
print("The global average LW cloud feedback is {val:.2f} W/m^2/K.".format(
    val=(ck.spat_avg(cld_lw)/dTS_glob_avg).mean()))

The global average SW cloud feedback is 0.38 W/m^2/K.
The global average LW cloud feedback is 0.03 W/m^2/K.


In [11]:
## the above is exactly as in Ty's tutorial https://github.com/tyfolino/climkern

## below, let's try recreating FP 14's figure for arctic vs tropical warming contributions, using the tutorial data.
## note that ctrl is 1xco2, pert is 2xco2

In [38]:
Planck_gm = ck.spat_avg(Planck)

Rs = {'Cloud':-(cld_lw+cld_sw),
      'Alb':-alb,
      'WV':-(q_lw+q_sw),
      'PL':Planck-Planck_gm,
      'LR':-LR,
      'CO2':-(erf.erf_lwas + erf.erf_swas)}

R_names = [x for x in Rs.keys()]

arctic_contribs, tropics_contribs = [], []

for name in R_names:
    arctic_contrib = ck.spat_avg(Rs[name].sel(lat=slice(66, 91)))/Planck_gm
    tropics_contrib = ck.spat_avg(Rs[name].sel(lat=slice(-23, 23)))/Planck_gm
    arctic_contribs.append(arctic_contrib.mean('time').values.item())
    tropics_contribs.append(tropics_contrib.mean('time').values.item())

arctic_dict = dict(zip(R_names, arctic_contribs))
tropics_dict = dict(zip(R_names, tropics_contribs))

both_dict = dict(zip(['Arctic', 'Tropics'],[arctic_dict, tropics_dict]))

In [40]:
import json
with open("test.json", "w") as json_file:
    # json.dump() writes the dictionary to the file
    json.dump(both_dict, json_file)

In [34]:
arctic_contribs

[0.43921284875197725,
 0.8042907737288948,
 0.2854508825730401,
 1.34580426647927,
 0.9187548615765672,
 0.5874702485621592]

In [35]:
tropics_contribs

[0.0980026008178569,
 0.005452404530866818,
 0.6358243062988901,
 -0.11177667168978989,
 -0.36270448600172706,
 0.49738685318747505]

In [36]:
import numpy as np